# Imports

In [ ]:
from Functions.Fcts_QC import calculate_outgrowth, normalize_groups, plate_bias_overview
from Functions.Fcts_Base import load_adata, save_adata
import seaborn as sns


%load_ext autoreload
%autoreload 2

# User Input

In [ ]:
load_dir = "PATH TO OUTPUT FROM 2_Filtering" # Path to .h5ad file containing the AnnData object.

# Set up plotting parameters
fig_param = {"axes.spines.right": False,
                  "axes.spines.top": False,
                  "axes.spines.left": True,
                  "axes.spines.bottom": True,
                  "pdf.fonttype": 42,
                  "font.size": 6,
                  "axes.labelsize": 6,
                  "axes.titlesize": 8,
                  "xtick.labelsize": 6,
                  "ytick.labelsize": 6,
                  "legend.fontsize": 6}

sns.set_theme(rc = fig_param, style = "ticks")

# Load & misc

In [ ]:
ad = load_adata(load_dir)

# Calculate outgrowth
Calculates the outgrowth information per well by taking all indicated wells in the .xls setup file into account, even if no organoids are detected during the segmentation.

In [ ]:
n_seeded = 9999 # Number of cells seeded at day 0 per well. Assumed to be constant.
ad = calculate_outgrowth(ad = ad, n_seeded = n_seeded)

In [ ]:
ad.uns["Outgrowth_DF"]

# Normalization
Performs z, robust, and minmax normalization. Area-associated features are log1p normalized to improve distribution.

In [ ]:
group_by = "Barcode" # The column name(s) in adata.obs to group the data by. Can be a list of str.
control = {"Medium": "Control"} # A dictionary specifying the control group for normalization. The key is the column name, and the value is the control group value. If None, each group is normalized independently.

ad = normalize_groups(
    ad, 
    group_by = group_by, 
    control = control)

# Plate overview to check for positional bias
Generate heatmaps illustrating plate bias based on specified features and experimental conditions. Values are z-scores of well averages normalized within conditions.

In [ ]:
features_to_plot = ["area", "R0__DAPI_mean"] # list of feature column names to visualize (mean values)
plate_size = 384 # 384 or 96 plate setup
plot_only_control = False # True or False. If True, only wells matching "control" are plotted.

plate_bias_overview(
    plt_features = features_to_plot,
    ad = ad,
    plate_size = plate_size,
    control_condition = control,
    control_only = plot_only_control
    )

# Save AD

In [ ]:
save_adata(ad, "3_QC")